## Parte 1: Python puro

En esta sección conviene enfocarse en tres preguntas:

1. ¿Cuál es la complejidad teórica?
2. ¿Qué implementación gana para entradas chicas?
3. ¿Qué parte del tiempo está en CPU y qué parte es overhead?

In [ ]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
from time import perf_counter
import cProfile
import pstats
import random
import statistics

from sqlalchemy import delete
from sqlmodel import select

from src.database.database import async_session_factory, create_db_and_tables
from src.models import Inventory, Person, User
from src.services.inventory import InventoryService
from src.services.persons import PersonService
from src.services.user import UserService

In [ ]:
Path("profiles").mkdir(exist_ok=True)
random.seed(42)
print("Entorno listo.")

In [ ]:
def benchmark_sync(func, repeats: int = 5):
    timings = []
    last_result = None
    for _ in range(repeats):
        start = perf_counter()
        last_result = func()
        timings.append(perf_counter() - start)
    return {
        "runs": timings,
        "mean": statistics.mean(timings),
        "min": min(timings),
        "max": max(timings),
        "last_result": last_result,
    }


async def benchmark_async(factory, repeats: int = 3):
    timings = []
    last_result = None
    for _ in range(repeats):
        start = perf_counter()
        last_result = await factory()
        timings.append(perf_counter() - start)
    return {
        "runs": timings,
        "mean": statistics.mean(timings),
        "min": min(timings),
        "max": max(timings),
        "last_result": last_result,
    }


def profile_sync(func, sort_by: str = "cumulative", limit: int = 20):
    profiler = cProfile.Profile()
    profiler.enable()
    result = func()
    profiler.disable()

    buffer = StringIO()
    stats = pstats.Stats(profiler, stream=buffer).sort_stats(sort_by)
    stats.print_stats(limit)
    print(buffer.getvalue())
    return result, stats


async def profile_async(coro_factory, sort_by: str = "cumulative", limit: int = 20):
    profiler = cProfile.Profile()
    profiler.enable()
    result = await coro_factory()
    profiler.disable()

    buffer = StringIO()
    stats = pstats.Stats(profiler, stream=buffer).sort_stats(sort_by)
    stats.print_stats(limit)
    print(buffer.getvalue())
    return result, stats


def summarize(label: str, report: dict) -> None:
    print(
        f"{label}: mean={report['mean']:.6f}s min={report['min']:.6f}s "
        f"max={report['max']:.6f}s runs={report['runs']}"
    )

### Ejemplo A: búsqueda lineal vs construir un `set`

Este ejemplo es útil para mostrar que una solución con mejor complejidad por consulta no siempre gana si el costo de preparación no se amortiza.

- Búsqueda lineal sobre lista: `O(n)` por consulta.
- Construir `set` y consultar: construcción `O(n)`, consulta promedio `O(1)`.

In [ ]:
def contains_linear(values: list[int], targets: list[int]) -> list[bool]:
    return [target in values for target in targets]


def contains_with_set(values: list[int], targets: list[int]) -> list[bool]:
    values_set = set(values)
    return [target in values_set for target in targets]


def build_membership_case(size: int, queries: int):
    values = list(range(size))
    targets = [size - 1 - (i % size) for i in range(queries)]
    return values, targets

In [ ]:
size = 100
queries = 1
values, targets = build_membership_case(size, queries)

report_linear = benchmark_sync(lambda: contains_linear(values, targets))
report_set = benchmark_sync(lambda: contains_with_set(values, targets))

summarize("lista_lineal", report_linear)
summarize("set_construido_en_cada_corrida", report_set)

In [ ]:
size = 10_000
queries = 2_000
values, targets = build_membership_case(size, queries)

report_linear = benchmark_sync(lambda: contains_linear(values, targets))
report_set = benchmark_sync(lambda: contains_with_set(values, targets))

summarize("lista_lineal", report_linear)
summarize("set_construido_en_cada_corrida", report_set)

### Ejemplo B: contar frecuencias con doble bucle vs diccionario

Acá el contraste es más clásico:

- doble bucle / conteo repetido: `O(n^2)`
- acumulación en diccionario: `O(n)`

Para tamaños chicos, la implementación peor puede no verse tan mal. Para tamaños grandes, la diferencia debería crecer rápido.

In [ ]:
def frequencies_naive(values: list[int]) -> dict[int, int]:
    uniques = []
    for value in values:
        if value not in uniques:
            uniques.append(value)

    counts = {}
    for value in uniques:
        count = 0
        for candidate in values:
            if candidate == value:
                count += 1
        counts[value] = count
    return counts


def frequencies_dict(values: list[int]) -> dict[int, int]:
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    return counts

In [ ]:
values_small = [i % 10 for i in range(100)]
values_large = [i % 50 for i in range(10_000)]

summarize("frecuencias_naive_small", benchmark_sync(lambda: frequencies_naive(values_small)))
summarize("frecuencias_dict_small", benchmark_sync(lambda: frequencies_dict(values_small)))
print()
summarize("frecuencias_naive_large", benchmark_sync(lambda: frequencies_naive(values_large), repeats=3))
summarize("frecuencias_dict_large", benchmark_sync(lambda: frequencies_dict(values_large), repeats=3))

### Profiling en Python puro

Estos ejemplos son mayormente **CPU-bound**. Si el tamaño crece, el tiempo debería concentrarse en el cómputo y en los recorridos sobre estructuras en memoria.

In [ ]:
_result, _stats = profile_sync(lambda: frequencies_naive(values_large), limit=25)